# Experiment Tracking with MLflow

Reproducibility is critical in ML. This notebook covers:
1. **Why track experiments** -- parameters, metrics, artifacts
2. **MLflow basics** -- logging runs, comparing experiments
3. **Model registry** overview

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

try:
    import mlflow
    import mlflow.sklearn
    HAS_MLFLOW = True
except ImportError:
    HAS_MLFLOW = False
    print('mlflow not installed -- pip install mlflow')

%matplotlib inline

## 1. Why Track Experiments?

Without tracking, ML development quickly becomes chaotic:
- Which hyperparameters produced the best model?
- What data preprocessing was applied?
- Can I reproduce last week's result?

**MLflow** provides:
- **Tracking**: log parameters, metrics, and artifacts
- **Projects**: package code for reproducibility
- **Models**: standard format for model packaging
- **Registry**: model versioning and lifecycle management

In [ ]:
# Prepare data
wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.25, random_state=42, stratify=wine.target
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. MLflow: Logging Experiments

In [ ]:
if HAS_MLFLOW:
    # Set experiment name
    mlflow.set_experiment('wine-classification')
    
    # Run multiple experiments with different hyperparameters
    param_grid = [
        {'n_estimators': 50, 'max_depth': 3},
        {'n_estimators': 100, 'max_depth': 5},
        {'n_estimators': 200, 'max_depth': None},
        {'n_estimators': 100, 'max_depth': 10},
    ]
    
    for params in param_grid:
        with mlflow.start_run():
            # Log parameters
            mlflow.log_params(params)
            
            # Train model
            rf = RandomForestClassifier(**params, random_state=42)
            rf.fit(X_train, y_train)
            y_pred = rf.predict(X_test)
            
            # Log metrics
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, average='weighted')
            mlflow.log_metric('accuracy', acc)
            mlflow.log_metric('f1_weighted', f1)
            
            # Log model
            mlflow.sklearn.log_model(rf, 'model')
            
            print(f"Params: {params} -> acc={acc:.4f}, f1={f1:.4f}")
    
    print("\nAll runs logged. View with: mlflow ui")
else:
    # Demonstrate the same logic without MLflow
    results = []
    param_grid = [
        {'n_estimators': 50, 'max_depth': 3},
        {'n_estimators': 100, 'max_depth': 5},
        {'n_estimators': 200, 'max_depth': None},
    ]
    for params in param_grid:
        rf = RandomForestClassifier(**params, random_state=42)
        rf.fit(X_train, y_train)
        acc = accuracy_score(y_test, rf.predict(X_test))
        results.append({**params, 'accuracy': acc})
        print(f"{params} -> accuracy={acc:.4f}")

In [ ]:
if HAS_MLFLOW:
    # Query logged runs
    experiment = mlflow.get_experiment_by_name('wine-classification')
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
    print(runs[['params.n_estimators', 'params.max_depth', 'metrics.accuracy', 'metrics.f1_weighted']])

In [ ]:
if HAS_MLFLOW:
    # Log an artifact (e.g., a plot)
    with mlflow.start_run():
        rf = RandomForestClassifier(n_estimators=200, random_state=42)
        rf.fit(X_train, y_train)
        
        # Create and save feature importance plot
        fig, ax = plt.subplots(figsize=(8, 5))
        importances = rf.feature_importances_
        idx = np.argsort(importances)
        ax.barh(range(len(idx)), importances[idx])
        ax.set_yticks(range(len(idx)))
        ax.set_yticklabels([wine.feature_names[i] for i in idx])
        ax.set_title('Feature Importances')
        plt.tight_layout()
        fig.savefig('feature_importance.png', dpi=100)
        plt.show()
        
        mlflow.log_artifact('feature_importance.png')
        mlflow.log_metric('accuracy', accuracy_score(y_test, rf.predict(X_test)))
        print('Artifact logged.')

## 3. Model Registry (Concept)

The MLflow **Model Registry** provides:
- **Versioning**: track model iterations
- **Stage transitions**: Staging -> Production -> Archived
- **Annotations**: add descriptions and tags

```python
# Register a model (requires a tracking server)
mlflow.register_model('runs:/<run_id>/model', 'WineClassifier')

# Transition stage
client = mlflow.tracking.MlflowClient()
client.transition_model_version_stage('WineClassifier', version=1, stage='Production')
```

## Key Takeaways

- **Always track experiments** -- even for exploratory work.
- MLflow logs **parameters, metrics, artifacts, and models** in a standardised way.
- The **Model Registry** manages the lifecycle from experimentation to production.
- Alternative tools: Weights & Biases, Neptune, DVC.

**Next:** Model packaging and serialisation.